# Phase 2 RQ2 — Relation Relatedness Assessment

Assess which MIMIC × OMOP relation pairs LLMs consider semantically related.
The 9 ground-truth pairs serve as the positive class; all other pairs are
negative.  Computes relation-level precision, recall, and F1.

In [ ]:
# Cell 1 — environment setup (MUST run before any thesis-extension import)
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

_cwd = Path(os.getcwd()).resolve()
for _candidate in [_cwd, _cwd.parent, _cwd / "thesis-extension"]:
    if (_candidate / "pipeline.py").exists():
        _thesis_root = _candidate
        break
else:
    raise RuntimeError(
        f"Cannot locate thesis-extension root from CWD={_cwd}."
    )

_notebooks_dir = _thesis_root / "notebooks"
_project_root = _thesis_root.parent

os.environ.setdefault("RESULTS_DIR", str(_thesis_root / "results"))
os.environ.setdefault("TEMPLATE_DIR", str(_thesis_root / "templates"))
os.environ.setdefault("LLM_PROVIDER", "anthropic")

load_dotenv(_thesis_root / ".env", override=False)

for _d in [str(_thesis_root), str(_notebooks_dir)]:
    if _d not in sys.path:
        sys.path.insert(0, _d)

print("thesis-extension root:", _thesis_root)
print("RESULTS_DIR          :", os.environ["RESULTS_DIR"])

In [ ]:
# Cell 2 — imports and mock-mode check
from config import config
from models import Parameters, RelationRelatednessResult, Side
from prompt_building import build_relatedness_prompts
from prompt_postprocessing import postprocess_relatedness_answers
from prompt_sending import send_prompts
from utils import ALL_MIMIC_TABLES, ALL_OMOP_TABLES, RELATION_PAIRS, load_relation

if not config["QUERY_LLM"]:
    print("WARNING: QUERY_LLM=False — running in mock mode. LLM will NOT be called.")

_model = (
    config["ANTHROPIC_MODEL"]
    if config["LLM_PROVIDER"] == "anthropic"
    else config["OPENAI_MODEL"]
)
print(f"Provider: {config['LLM_PROVIDER']}, Model: {_model}")

In [ ]:
# Cell 3 — load ALL MIMIC and OMOP relations from schema_documentations/
mimic_relations = []
for name in ALL_MIMIC_TABLES:
    try:
        rel = load_relation(name, "MIMIC", Side.SOURCE)
        mimic_relations.append(rel)
    except FileNotFoundError as exc:
        print(f"  SKIP MIMIC/{name}: {exc}")

omop_relations = []
for name in ALL_OMOP_TABLES:
    try:
        rel = load_relation(name, "OMOP", Side.TARGET)
        omop_relations.append(rel)
    except FileNotFoundError as exc:
        print(f"  SKIP OMOP/{name}: {exc}")

print(f"Loaded {len(mimic_relations)} MIMIC relations, {len(omop_relations)} OMOP relations.")
print(f"Cartesian product size: {len(mimic_relations) * len(omop_relations)} pairs.")

In [ ]:
# Cell 4 — run relatedness assessment for all MIMIC × OMOP pairs
# GT pairs set for fast lookup
_GT_PAIRS = {(src_name, tgt_name) for src_name, _, tgt_name, _ in RELATION_PAIRS}

all_results: list[RelationRelatednessResult] = []
pair_count = 0

for src in mimic_relations:
    for tgt in omop_relations:
        params = Parameters(
            source_relation=src,
            target_relation=tgt,
            llm_model=_model,
        )
        prompts = build_relatedness_prompts(params)
        answers = send_prompts(params, prompts)
        parsed  = postprocess_relatedness_answers(answers)

        if parsed:
            # postprocess returns empty relation names; fill them in
            rr = parsed[0]
            rr.source_relation_name = src.name
            rr.target_relation_name = tgt.name
            all_results.append(rr)
        else:
            # Mock mode or unparseable answer — default to not related
            all_results.append(
                RelationRelatednessResult(
                    source_relation_name=src.name,
                    target_relation_name=tgt.name,
                    related=False,
                    confidence="low",
                    reasoning="no parseable answer",
                )
            )
        pair_count += 1

print(f"Processed {pair_count} pairs, got {sum(r.related for r in all_results)} 'related' decisions.")

In [ ]:
# Cell 5 — relation-level evaluation
predicted_related = {
    (r.source_relation_name, r.target_relation_name)
    for r in all_results
    if r.related
}

tp = predicted_related & _GT_PAIRS
fp = predicted_related - _GT_PAIRS
fn = _GT_PAIRS - predicted_related

precision = len(tp) / len(predicted_related) if predicted_related else 0.0
recall    = len(tp) / len(_GT_PAIRS) if _GT_PAIRS else 0.0
f1        = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0 else 0.0
)

print("=== Relation-level evaluation ===")
print(f"  Ground truth positives : {len(_GT_PAIRS)}")
print(f"  Predicted related      : {len(predicted_related)}")
print(f"  True  positives        : {len(tp)}")
print(f"  False positives        : {len(fp)}")
print(f"  False negatives        : {len(fn)}")
print(f"  Precision : {precision:.3f}")
print(f"  Recall    : {recall:.3f}")
print(f"  F1        : {f1:.3f}")

if fp:
    print("\nFalse positives (LLM thinks related, not in GT):")
    for src_n, tgt_n in sorted(fp):
        print(f"  {src_n} -> {tgt_n}")

if fn:
    print("\nFalse negatives (GT related, LLM missed):")
    for src_n, tgt_n in sorted(fn):
        print(f"  {src_n} -> {tgt_n}")

In [ ]:
# Cell 6 — save results
import json
import pandas as pd

out_dir = Path(os.environ["RESULTS_DIR"])
out_dir.mkdir(parents=True, exist_ok=True)

# Save per-pair results
rows = [
    {
        "source": r.source_relation_name,
        "target": r.target_relation_name,
        "related": r.related,
        "confidence": r.confidence,
        "reasoning": r.reasoning,
        "is_gt_pair": (r.source_relation_name, r.target_relation_name) in _GT_PAIRS,
    }
    for r in all_results
]
rel_df = pd.DataFrame(rows)
csv_path = out_dir / "relatedness_results.csv"
rel_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

# Save metrics
metrics = {
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "true_positives": len(tp),
    "false_positives": len(fp),
    "false_negatives": len(fn),
    "total_pairs": pair_count,
    "predicted_related": len(predicted_related),
    "gt_pairs": len(_GT_PAIRS),
}
metrics_path = out_dir / "relatedness_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(f"Saved: {metrics_path}")